# Lakelink Fiber Network Scenario Twin: synthetic data generator
The single generator notebook for all P1-GEN-* tasks (D-006, data_contracts.md section 0). Target `cdm_tmforum.network_scenario_twin`; tag `telco_project: network_scenario_twin`; seed 4471; run `nst-sim-001`.

Sections (one per task): P1-GEN-REFERENCE (done) | P1-GEN-TOPOLOGY | P1-GEN-SERVICE | P1-GEN-TELEMETRY | P1-GEN-OPS | P1-GEN-COPPER.

Rules: REUSE FIRST. FK values come only from existing parent tables. Company name is always Lakelink Fiber. Customer names are fictional.

In [0]:
import random, csv
from pyspark.sql import functions as F
CAT, SCH = "cdm_tmforum", "network_scenario_twin"
FQ = lambda t: f"{CAT}.{SCH}.{t}"
SEED, RUN_ID = 4471, "nst-sim-001"
ROOT = "/Workspace/Users/stephen.hage@databricks.com/network-scenario-twin"

def publish(table, df, table_comment, col_comments):
    """Overwrite a twin table, then apply table/column comments and governed tags."""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FQ(table))
    esc = lambda s: s.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {FQ(table)} IS '{esc(table_comment)}'")
    for c, v in col_comments.items():
        spark.sql(f"ALTER TABLE {FQ(table)} ALTER COLUMN {c} COMMENT '{esc(v)}'")
    spark.sql(f"ALTER TABLE {FQ(table)} SET TAGS ('domain'='operations')")
    try:
        spark.sql(f"ALTER TABLE {FQ(table)} SET TAGS ('telco_project'='network_scenario_twin')")
    except Exception as e:  # governed tag policy lacks the value; back-fill in P1-TAG-REAPPLY
        if "UC_TAG_POLICY_VALUE_NOT_ALLOWED" not in str(e): raise
        UNTAGGED.append(table)

UNTAGGED = []  # tables published without the telco_project tag (P1-TAG-POLICY-VALUE-RESTORE)

LINEAGE = {"source_table": "Lineage: fully qualified parent table (null for SYNTHETIC).",
           "source_id": "Lineage: parent key value as string (null for SYNTHETIC).",
           "origin": "Data origin: REAL, MIXED or SYNTHETIC.",
           "simulation_run_id": "Generator run id (nst-sim-001)."}

In [0]:
# EXTEND tmf_resource.resource_specification: 20 non-deprecated transport specs, lowest ids first
type_plan = ["fiber_span"]*4 + ["optical_node"]*4 + ["site"]*2 + ["wavelength"]*4 + ["port"]*4 + ["och"]*2
src = (spark.table(f"{CAT}.tmf_resource.resource_specification")
       .where("status <> 'deprecated' AND max_bandwidth_mbps IS NOT NULL AND technology_domain IN ('sdh','mpls','ftth','fixed','gpon')")
       .orderBy("resource_specification_id").limit(20)
       .select("resource_specification_id", "max_bandwidth_mbps").collect())
assert len(src) == 20
label = {"fiber_span": "Long-Haul Fiber Span", "optical_node": "ROADM Optical Node", "site": "Network Site",
         "wavelength": "Coherent Wavelength", "port": "Client Port", "och": "Optical Channel"}
spec_rows = []
for i, (r, t) in enumerate(zip(src, type_plan)):
    cap = round(float(r.max_bandwidth_mbps) / 1000.0, 2)
    spec_rows.append((int(r.resource_specification_id), f"Lakelink Fiber {int(round(cap))}G {label[t]} v{i%3+1}", t, cap,
                      f"{CAT}.tmf_resource.resource_specification", str(r.resource_specification_id), "MIXED", RUN_ID))
spec_df = spark.createDataFrame(spec_rows, "spec_id BIGINT, name STRING, type STRING, capacity_gbps DOUBLE, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("resource_specification", spec_df,
    "Lakelink Fiber Network Scenario Twin: resource specification catalog. Grain: 1 row per spec (20 rows). EXTEND of tmf_resource.resource_specification; origin MIXED; run nst-sim-001. No injected anomaly.",
    {"spec_id": "PK. Resource specification id; equals tmf_resource.resource_specification.resource_specification_id.",
     "name": "Lakelink Fiber display name of the spec (no vendor branding).",
     "type": "Twin spec type: fiber_span, optical_node, site, wavelength, port, och.",
     "capacity_gbps": "Nominal capacity in Gbps = source max_bandwidth_mbps / 1000.", **LINEAGE})

In [0]:
# Ratified reference file (data/corridor_reference.csv); endpoints FK -> fiber_capital_intelligence.geographic_site
with open(f"{ROOT}/data/corridor_reference.csv") as f:
    rows = list(csv.DictReader(f))
nz = lambda v: v if v not in ("", None) else None
fl = lambda v: float(v) if nz(v) else None
cor_rows = [(r["corridor_id"], r["corridor_name"], r["route_type"], int(r["site_a_id"]), int(r["site_z_id"]), fl(r["distance_km"]),
             fl(r["source_distance_km"]), int(r["fiber_count"]), fl(r["capacity_gbps"]), nz(r["diverse_of"]), fl(r["baseline_util_pct"]),
             fl(r["end_util_pct"]), nz(r["injected_anomaly"]), nz(r["source_table"]), nz(r["source_id"]), r["origin"], RUN_ID) for r in rows]
cor_df = spark.createDataFrame(cor_rows,
    "corridor_id STRING, corridor_name STRING, route_type STRING, site_a_id INT, site_z_id INT, distance_km DOUBLE, source_distance_km DOUBLE, "
    "fiber_count INT, capacity_gbps DOUBLE, diverse_of STRING, baseline_util_pct DOUBLE, end_util_pct DOUBLE, injected_anomaly STRING, "
    "source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("corridor", cor_df,
    "Lakelink Fiber Network Scenario Twin: fiber corridors (long-haul and metro). Grain: 1 row per corridor (40 rows: 33 MIXED from fiber_capital_intelligence.silver_corridor, 7 SYNTHETIC). Injected anomalies: DEN-SLC RapidRoute exhaustion (55% to 85% utilization, ~120 days to 95%), CHI-DEN / CHI-DEN-D2 sold as diverse but sharing SRLG-88.",
    {"corridor_id": "PK. Corridor id: <A>-<Z> alphabetical city codes, <CITY>-MET for metro rings, <base>-D2 for diverse routes.",
     "corridor_name": "Lakelink Fiber corridor display name (DEN-SLC = DEN-SLC RapidRoute).",
     "route_type": "long_haul or metro.",
     "site_a_id": "FK -> fiber_capital_intelligence.geographic_site.geographic_site_id (A endpoint).",
     "site_z_id": "FK -> fiber_capital_intelligence.geographic_site.geographic_site_id (Z endpoint).",
     "distance_km": "Route length in km. Long-haul = great-circle x 1.25 (diverse -D2 x1.12 more); metro = source value.",
     "source_distance_km": "Raw silver_corridor.total_distance_km (sum of segment lengths). Lineage only; do not use for analysis.",
     "fiber_count": "Fiber strand count on the corridor.",
     "capacity_gbps": "Corridor lit capacity in Gbps (DEN-SLC = 9600).",
     "diverse_of": "FK -> corridor.corridor_id of the primary route this diverse route protects; null otherwise.",
     "baseline_util_pct": "Generator target utilization (0-100) at 2026-08-18.",
     "end_util_pct": "Generator target utilization (0-100) at 2026-11-16.",
     "injected_anomaly": "Demo anomaly label: DEN-SLC_EXHAUSTION or SRLG-88_DIVERSE_PAIR; null otherwise.", **LINEAGE})

In [0]:
# customer_id drawn ONLY from the audited pool (P1-CUSTOMER-NAME-AUDIT); fictional display names, deterministic seed
pool = spark.table(FQ("dq_customer_candidate_pool")).orderBy("customer_id").collect()
rng = random.Random(SEED)
picked = sorted(rng.sample(pool, 120), key=lambda r: r.customer_id)
ADJ = ["Aspen","Bluestem","Cedar Ridge","Copperleaf","Driftwood","Ember","Foxglove","Granite Peak","Harborline","Ironwood","Juniper","Kestrel",
       "Larkspur","Meridian Plains","Northwind","Oakhollow","Prairie Sky","Quarry Hill","Redfern","Silverbrook","Tamarack","Umber","Valewood",
       "Willowmere","Yarrow","Zephyr","Amberfield","Briar Creek","Cobalt Mesa","Dunmore","Elmstead","Fernhaven","Glacier Run","Highgate"]
NOUN = ["Logistics","Health Partners","Data Systems","Manufacturing","Credit Union","Media Group","Analytics","Energy Cooperative",
        "Freight","Biosciences","Retail Group","Learning Network","Water District","Hospitality","Robotics","Insurance Mutual",
        "Cloud Works","Foods","Engineering","Capital Partners"]
SUF = ["", " Co.", " Group", " Holdings", " Collective"]
DENY = ["lakelink","lumen","centurylink","level 3","att","at&t","verizon","comcast","zayo","cogent","google","amazon","microsoft","meta",
        "apple","ibm","oracle","cisco","t-mobile","sprint","charter","frontier","windstream","crown castle","equinix","nvidia"]
names, cust = set(), []
for r in picked:
    while True:
        n = f"{rng.choice(ADJ)} {rng.choice(NOUN)}{rng.choice(SUF)}"
        if n not in names and not any(d in n.lower() for d in DENY if len(d) > 3) and not any(w in DENY for w in n.lower().split()):
            break
    names.add(n); cust.append((r, n))
ids = [r.customer_id for r, _ in cust]
dual = set(rng.sample(ids, 12))                                   # SRLG-88 Dual Path customers
iowa = set(rng.sample([i for i in ids if i not in dual], 3))       # Iowa wire center IA-WC-01 high-value customers
remaining = [i for i in ids if i not in dual and i not in iowa]
hyper = set(rng.sample(remaining, 6)); gov = set(rng.sample([i for i in remaining if i not in hyper], 8))
def seg(r):
    if r.customer_id in hyper: return "hyperscaler"
    if r.customer_id in gov: return "government"
    if r.type == "wholesale": return "wholesale"
    return "smb" if r.segment_classification == "sme" else "enterprise"
cust_rows = []
for r, n in cust:
    s, cid = seg(r), r.customer_id
    naas = cid in dual or s in ("hyperscaler", "enterprise") and rng.random() < 0.45 or rng.random() < 0.15
    cust_rows.append((cid, n, s, f"{r.segment_classification}/{r.type}", bool(naas), cid in dual,
                      "IA-WC-01" if cid in iowa else None, cid in iowa, r.source_name_code,
                      f"{CAT}.tmf_customer.customer", str(cid), "MIXED", RUN_ID))
cust_df = spark.createDataFrame(cust_rows,
    "customer_id BIGINT, name STRING, segment STRING, source_segment STRING, naas_flag BOOLEAN, dual_path_flag BOOLEAN, copper_wc_id STRING, "
    "high_value_flag BOOLEAN, source_name_code STRING, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("customer", cust_df,
    "Lakelink Fiber Network Scenario Twin: business customers. Grain: 1 row per customer (120 rows). customer_id drawn only from dq_customer_candidate_pool (-> tmf_customer.customer); names are fictional. Injected: 12 Dual Path customers (dual_path_flag) on SRLG-88 spans; 3 high-value customers on Iowa wire center IA-WC-01.",
    {"customer_id": "PK and FK -> tmf_customer.customer.customer_id (via dq_customer_candidate_pool).",
     "name": "Fictional business display name (deterministic, seed 4471, checked against a real-brand denylist). Never a real company.",
     "segment": "Twin segment: enterprise, wholesale, hyperscaler, government, smb.",
     "source_segment": "Source segment_classification/type from tmf_customer.customer, kept for lineage.",
     "naas_flag": "TRUE if the customer buys NaaS / on-demand services.",
     "dual_path_flag": "TRUE for the 12 Dual Path customers whose diverse paths ride S-4471 and S-4472 (shared SRLG-88).",
     "copper_wc_id": "FK -> copper_wire_center.wc_id; set only for the 3 Iowa high-value customers (IA-WC-01).",
     "high_value_flag": "High-value customer. TRUE for the 3 Iowa customers now; top-decile MRR customers added in P1-GEN-SERVICE.",
     "source_name_code": "Opaque source name code from tmf_customer.customer (lineage only; not for display).", **LINEAGE})

In [0]:
display(spark.sql(f"""
SELECT 'spec_rows' k, count(*) v FROM {FQ('resource_specification')}
UNION ALL SELECT 'spec_fk_orphan', count(*) FROM {FQ('resource_specification')} s LEFT ANTI JOIN {CAT}.tmf_resource.resource_specification p ON s.spec_id=p.resource_specification_id
UNION ALL SELECT 'cor_rows', count(*) FROM {FQ('corridor')}
UNION ALL SELECT 'cor_site_orphan', count(*) FROM (SELECT site_a_id s FROM {FQ('corridor')} UNION ALL SELECT site_z_id FROM {FQ('corridor')}) c LEFT ANTI JOIN {CAT}.fiber_capital_intelligence.geographic_site g ON c.s=g.geographic_site_id
UNION ALL SELECT 'denslc_rapidroute', count(*) FROM {FQ('corridor')} WHERE corridor_id='DEN-SLC' AND corridor_name='DEN-SLC RapidRoute' AND capacity_gbps=9600
UNION ALL SELECT 'cust_rows', count(*) FROM {FQ('customer')}
UNION ALL SELECT 'cust_parent_orphan', count(*) FROM {FQ('customer')} c LEFT ANTI JOIN {CAT}.tmf_customer.customer p USING (customer_id)
UNION ALL SELECT 'dual_path', count_if(dual_path_flag) FROM {FQ('customer')}
UNION ALL SELECT 'iowa_hv', count_if(copper_wc_id='IA-WC-01' AND high_value_flag) FROM {FQ('customer')}
"""))

In [0]:
# 600 spans (EXTEND provable_resilience_intelligence.physical_resource_span) + 150 SYNTHETIC nodes + 50 sites (EXTEND fiber_capital_intelligence.geographic_site)
# One source SRLG/conduit group per corridor (remapped SRLG-01..SRLG-40); a 41st group becomes SRLG-88 = {S-4471 on CHI-DEN, S-4472 on CHI-DEN-D2}
import math
SPAN_SRC = f"{CAT}.provable_resilience_intelligence.physical_resource_span"
SRLG_SRC = f"{CAT}.provable_resilience_intelligence.derived_srlg_shared_conduit"
SITE_SRC = f"{CAT}.fiber_capital_intelligence.geographic_site"
rng = random.Random(SEED)
cors = spark.table(FQ("corridor")).orderBy("corridor_id").collect()
spec = {t: [r.spec_id for r in spark.table(FQ("resource_specification")).where(f"type='{t}'").orderBy("spec_id").collect()]
        for t in ("fiber_span", "optical_node", "site")}

def lr_alloc(total, weights, floor):
    """Largest-remainder allocation of total across weights with a per-item floor."""
    base = [floor]*len(weights); rem = total - floor*len(weights); sw = sum(weights)
    raw = [rem*w/sw for w in weights]; add = [int(x) for x in raw]
    order = sorted(range(len(weights)), key=lambda i: (-(raw[i]-add[i]), i))
    for i in order[: rem - sum(add)]: add[i] += 1
    return [b+a for b, a in zip(base, add)]

dist = [c.distance_km for c in cors]
n_spans = lr_alloc(600, dist, 4)
lh = [i for i, c in enumerate(cors) if c.route_type == "long_haul"]
inline = [0]*len(cors)
for i, k in zip(lh, lr_alloc(70, [dist[i] for i in lh], 0)): inline[i] = k
assert sum(n_spans) == 600 and sum(inline) + 2*len(cors) == 150

grp = spark.sql(f"""SELECT d.srlg_id, p.conduit_id, collect_list(p.span_id) spans, count(*) n
  FROM {SRLG_SRC} d JOIN {SPAN_SRC} p USING (span_id) GROUP BY 1,2 ORDER BY n DESC, srlg_id""").collect()
need_order = sorted(range(len(cors)), key=lambda i: (-n_spans[i], cors[i].corridor_id))
cor_grp = {i: grp[j] for j, i in enumerate(need_order)}
g88 = grp[len(cors)]
for i in range(len(cors)): assert cor_grp[i].n >= n_spans[i]
twin_srlg = {i: f"SRLG-{k+1:02d}" for k, i in enumerate(range(len(cors)))}  # SRLG-01..SRLG-40; 88 reserved
span_len = {r.span_id: r.length_km for r in spark.table(SPAN_SRC).select("span_id","length_km").collect()}

end_sites = sorted({c.site_a_id for c in cors} | {c.site_z_id for c in cors})
cities = ['Denver','Salt Lake City','Chicago','Dallas','Ashburn','Seattle','Atlanta','Phoenix']
fill = [r.geographic_site_id for r in spark.table(SITE_SRC).where(F.col("city").isin(cities)).orderBy("geographic_site_id").collect()
        if r.geographic_site_id not in end_sites]
site_ids = sorted(end_sites + fill[: 50-len(end_sites)])
assert len(site_ids) == 50
P = []
for k, s in enumerate(site_ids):
    P.append((f"SITE-{s}", "site", spec["site"][k % 2], None, None, None, None, s, None, None, None, "in_service", SITE_SRC, str(s), "MIXED"))
nid, spans = 1001, []
for i, c in enumerate(cors):
    m = inline[i] + 2; nodes = []
    for p in range(m):
        home = c.site_a_id if p < m/2 else c.site_z_id
        nodes.append(f"N-{nid}")
        P.append((f"N-{nid}", "node", spec["optical_node"][nid % 4], c.corridor_id, None, None, f"SITE-{home}", None, None, None, None, "in_service", None, None, "SYNTHETIC"))
        nid += 1
    g = cor_grp[i]; src = sorted(g.spans)[: n_spans[i]]
    for k in range(n_spans[i]):
        h = k*(m-1)//n_spans[i]
        spans.append([c.corridor_id, nodes[h], nodes[h+1], twin_srlg[i], g.conduit_id, src[k]])
assert nid == 1151 and len(spans) == 600
ids = [f"S-{4001+k}" for k in range(600)]
def first_idx(cid):
    return next(k for k, s in enumerate(spans) if s[0] == cid)
a, b = first_idx("CHI-DEN"), first_idx("CHI-DEN-D2")
pa = ids.index("S-4471"); ids[a], ids[pa] = ids[pa], ids[a]
pb = ids.index("S-4472"); ids[b], ids[pb] = ids[pb], ids[b]
src88 = sorted(g88.spans)[:2]
for k, src_k in ((a, src88[0]), (b, src88[1])):
    spans[k][3], spans[k][4], spans[k][5] = "SRLG-88", g88.conduit_id, src_k
for k, s in enumerate(spans):
    P.append((ids[k], "span", spec["fiber_span"][k % 4], s[0], s[1], s[2], None, None, s[3], s[4], float(span_len[s[5]]), "in_service", SPAN_SRC, s[5], "MIXED"))
P = [r + (RUN_ID,) for r in P]
pr_schema = ("resource_id STRING, type STRING, spec_id LONG, corridor_id STRING, site_a_id STRING, site_z_id STRING, parent_site_id STRING, "
             "geographic_site_id INT, srlg STRING, conduit_id STRING, length_km DOUBLE, status STRING, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("physical_resource", spark.createDataFrame(P, pr_schema),
  "Lakelink Fiber Network Scenario Twin: physical network resources. Grain: 1 row per resource (800 = 600 fiber spans S-4001..S-4600, 150 optical nodes N-1001..N-1150, "
  "50 sites SITE-<geographic_site_id>). Spans EXTEND provable_resilience_intelligence.physical_resource_span (MIXED; source span_id in source_id, SRLG/conduit from "
  "derived_srlg_shared_conduit remapped to SRLG-01..SRLG-40, one SRLG per corridor); nodes SYNTHETIC; sites EXTEND fiber_capital_intelligence.geographic_site (MIXED). "
  "Injected anomaly SRLG-88: S-4471 (CHI-DEN) and S-4472 (CHI-DEN-D2, sold as diverse of CHI-DEN) share SRLG-88 and one conduit. Seed 4471, run nst-sim-001.",
  {"resource_id": "PK. Span S-####, node N-####, or site SITE-<geographic_site_id>.",
   "type": "Resource type: span, node or site.",
   "spec_id": "FK -> resource_specification.spec_id (fiber_span, optical_node or site spec).",
   "corridor_id": "FK -> corridor.corridor_id. Set for spans and nodes; null for sites.",
   "site_a_id": "Span A-end node: FK -> physical_resource.resource_id (type node). Null for nodes and sites.",
   "site_z_id": "Span Z-end node: FK -> physical_resource.resource_id (type node). Null for nodes and sites.",
   "parent_site_id": "Node home site: FK -> physical_resource.resource_id (type site). Null for spans and sites.",
   "geographic_site_id": "Sites only: FK -> fiber_capital_intelligence.geographic_site.geographic_site_id.",
   "srlg": "Shared-risk link group of the span conduit (SRLG-##). SRLG-88 = injected diverse-route shared risk (S-4471, S-4472). Null for nodes and sites.",
   "conduit_id": "Span conduit id from the source span group; spans in one SRLG share one conduit. Null for nodes and sites.",
   "length_km": "Span length in km from source physical_resource_span.length_km. Null for nodes and sites.",
   "status": "Lifecycle status: in_service, planned or retired.", **LINEAGE})

In [0]:
# EXTEND tmf_resource.logical_resource: OCh <- logical_channel (600, 1 per span), wavelength <- logical_link (1,300), port <- logical_port (600)
LR_SRC = f"{CAT}.tmf_resource.logical_resource"
def take(rt, n):
    rows = spark.table(LR_SRC).where(f"resource_type='{rt}'").orderBy("logical_resource_id").limit(n).select("logical_resource_id","physical_resource_id").collect()
    assert len(rows) == n; return rows
och_src, wl_src, port_src = take("logical_channel", 600), take("logical_link", 1300), take("logical_port", 600)
cmap = {c.corridor_id: c for c in cors}
span_rows = sorted([r for r in P if r[1] == "span"], key=lambda r: r[0])
extra = set(rng.sample(range(600), 100))
L, wi = [], 0
for k, s in enumerate(span_rows):
    c = cmap[s[3]]; u = c.end_util_pct/100.0
    o = och_src[k]; cap = float(c.capacity_gbps)
    L.append((o.logical_resource_id, None, "och", s[0], cap, round(cap*u, 2), o.physical_resource_id, LR_SRC, str(o.logical_resource_id)))
    first_wl = None
    for _ in range(3 if k in extra else 2):
        w_ = wl_src[wi]; wi += 1
        first_wl = first_wl or w_.logical_resource_id
        L.append((w_.logical_resource_id, o.logical_resource_id, "wavelength", s[0], 400.0, round(min(400.0, 400*u*rng.uniform(0.9, 1.1)), 2), w_.physical_resource_id, LR_SRC, str(w_.logical_resource_id)))
    pt = port_src[k]
    L.append((pt.logical_resource_id, first_wl, "port", s[0], 100.0, round(min(100.0, 100*u*rng.uniform(0.9, 1.1)), 2), pt.physical_resource_id, LR_SRC, str(pt.logical_resource_id)))
assert wi == 1300 and len(L) == 2500
L = [r + ("MIXED", RUN_ID) for r in L]
lr_df = spark.createDataFrame(L, "resource_id LONG, parent_id LONG, type STRING, span_id STRING, capacity_gbps DOUBLE, utilized_gbps DOUBLE, "
                                 "source_physical_resource_id LONG, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("logical_resource", lr_df,
  "Lakelink Fiber Network Scenario Twin: logical transport resources riding the fiber spans. Grain: 1 row per logical resource (2,500 = 600 OCh, 1,300 wavelengths, 600 client ports). "
  "EXTEND tmf_resource.logical_resource (resource_id = logical_resource_id; OCh from logical_channel, wavelength from logical_link, port from logical_port), origin MIXED. "
  "Hierarchy: port -> wavelength -> OCh -> span. utilized_gbps is the latest snapshot at anchor 2026-11-16 (corridor end_util_pct). Seed 4471, run nst-sim-001.",
  {"resource_id": "PK. Equals tmf_resource.logical_resource.logical_resource_id.",
   "parent_id": "FK -> logical_resource.resource_id: wavelength -> its OCh, port -> its wavelength; null for OCh.",
   "type": "Logical type: och, wavelength or port.",
   "span_id": "realizesOn FK -> physical_resource.resource_id (type span).",
   "capacity_gbps": "Capacity in Gbps (> 0): OCh = corridor capacity, wavelength = 400, port = 100.",
   "utilized_gbps": "Utilized Gbps at anchor 2026-11-16; 0 <= utilized_gbps <= capacity_gbps.",
   "source_physical_resource_id": "Source tmf_resource.logical_resource.physical_resource_id, lineage only (not a twin FK).", **LINEAGE})

In [0]:
pr, lr = FQ('physical_resource'), FQ('logical_resource')
display(spark.sql(f"""
SELECT 'pr_rows' k, count(*) v FROM {pr}
UNION ALL SELECT 'spans', count_if(type='span') FROM {pr}
UNION ALL SELECT 'srlg88_exact', count_if(srlg='SRLG-88' AND ((resource_id='S-4471' AND corridor_id='CHI-DEN') OR (resource_id='S-4472' AND corridor_id='CHI-DEN-D2'))) FROM {pr}
UNION ALL SELECT 'srlg88_spans', count_if(srlg='SRLG-88') FROM {pr}
UNION ALL SELECT 'srlg88_conduits', count(distinct conduit_id) FROM {pr} WHERE srlg='SRLG-88'
UNION ALL SELECT 'span_end_orphan', count(*) FROM (SELECT site_a_id e FROM {pr} WHERE type='span' UNION ALL SELECT site_z_id FROM {pr} WHERE type='span') x LEFT ANTI JOIN (SELECT resource_id FROM {pr} WHERE type='node') n ON x.e=n.resource_id
UNION ALL SELECT 'site_geo_orphan', count(*) FROM {pr} p LEFT ANTI JOIN {CAT}.fiber_capital_intelligence.geographic_site g ON p.geographic_site_id=g.geographic_site_id WHERE p.type='site'
UNION ALL SELECT 'span_source_orphan', count(*) FROM {pr} p LEFT ANTI JOIN {CAT}.provable_resilience_intelligence.physical_resource_span s ON p.source_id=s.span_id WHERE p.type='span'
UNION ALL SELECT 'lr_rows', count(*) FROM {lr}
UNION ALL SELECT 'lr_src_orphan', count(*) FROM {lr} l LEFT ANTI JOIN {CAT}.tmf_resource.logical_resource s ON l.resource_id=s.logical_resource_id
UNION ALL SELECT 'lr_span_orphan', count(*) FROM {lr} l LEFT ANTI JOIN (SELECT resource_id FROM {pr} WHERE type='span') s ON l.span_id=s.resource_id
UNION ALL SELECT 'lr_cap_bad', count_if(capacity_gbps<=0 OR utilized_gbps<0 OR utilized_gbps>capacity_gbps) FROM {lr}
"""))

In [0]:
# EXTEND tmf_service.service (1,500 distinct ACTIVE services) and tmf_customer.service_level_agreement (1:1).
# SRLG-88: first slot of each of the 12 dual_path_flag customers = active dual_path wavelength, primary on S-4471, protect on S-4472.
# No other service touches S-4471/S-4472. SLA ids: 1,226 matched to the twin customer, 274 reused from other source SLAs (source_customer_id kept).
import random
from pyspark.sql import types as T
rng = random.Random(SEED + 6)
SRLG_SPANS = {"S-4471", "S-4472"}
svc_src = (spark.table(f"{CAT}.tmf_service.service").where("status='active' AND monthly_recurring_charge > 0")
           .orderBy(F.xxhash64("service_id", F.lit(SEED)), "service_id").limit(1500)
           .select("service_id", "party_id", "monthly_recurring_charge").collect())
assert len(svc_src) == 1500
cust = {r.customer_id: r for r in spark.table(FQ("customer")).collect()}
SLA_SRC = spark.table(f"{CAT}.tmf_customer.service_level_agreement")
COLS = ["service_level_agreement_id", "customer_id", "latency_threshold_milliseconds", "penalty_cap_percentage"]
sla_match = SLA_SRC.join(spark.table(FQ("customer")).select("customer_id"), "customer_id").select(*COLS).orderBy("customer_id", "service_level_agreement_id").collect()
slots = [(r.customer_id, r) for r in sla_match]
extra_sla = (SLA_SRC.join(spark.table(FQ("customer")).select("customer_id"), "customer_id", "left_anti")
             .orderBy(F.xxhash64("service_level_agreement_id", F.lit(SEED)), "service_level_agreement_id").limit(1500 - len(slots)).select(*COLS).collect())
cids = sorted(cust)
slots += [(rng.choice(cids), r) for r in extra_sla]
assert len(slots) == 1500 and len({s[1].service_level_agreement_id for s in slots}) == 1500
svcs = list(svc_src); rng.shuffle(svcs)
spans = {r.resource_id: r for r in spark.table(FQ("physical_resource")).where("type='span'").collect()}
cor = {r.corridor_id: r for r in spark.table(FQ("corridor")).collect()}
lrs = spark.table(FQ("logical_resource")).where("type IN ('wavelength','port')").select("resource_id","type","span_id").orderBy("resource_id").collect()
lr_by = {}
for l in lrs: lr_by.setdefault((l.type, spans[l.span_id].corridor_id), []).append(l)
lr_pool = {t: [l for l in lrs if l.type == t and l.span_id not in SRLG_SPANS] for t in ("wavelength","port")}
def diverse_corridors(cid):
    c = cor[cid]; out = [k for k, v in cor.items() if v.diverse_of == cid]
    if c.diverse_of: out.append(c.diverse_of)
    return out
def pick_protect(ltype, primary):
    p_span = spans[primary.span_id]; cands = []
    for dc in diverse_corridors(p_span.corridor_id):
        cands += [l for l in lr_by.get((ltype, dc), []) if l.span_id not in SRLG_SPANS]
    if not cands:
        cands = [l for l in lr_pool[ltype] if spans[l.span_id].corridor_id != p_span.corridor_id and spans[l.span_id].srlg != p_span.srlg]
    return rng.choice(cands)
dp_cust = sorted(c for c, r in cust.items() if r.dual_path_flag); assert len(dp_cust) == 12
wl71 = sorted([l for l in lrs if l.type=="wavelength" and l.span_id=="S-4471"], key=lambda l: l.resource_id)
wl72 = sorted([l for l in lrs if l.type=="wavelength" and l.span_id=="S-4472"], key=lambda l: l.resource_id)
anchor_slot = {}
for i, (c, s) in enumerate(slots):
    if c in dp_cust and c not in anchor_slot: anchor_slot[c] = i
anchor_idx = {v: k for k, v in anchor_slot.items()}
svc_rows, sla_rows = [], []
for i, ((cid, sla), src) in enumerate(zip(slots, svcs)):
    c = cust[cid]
    if i in anchor_idx:
        j = sorted(anchor_slot).index(cid)
        stype, scheme, prim, prot, status = "wavelength", "dual_path", wl71[j % len(wl71)], wl72[j % len(wl72)], "active"
    else:
        stype = rng.choices(["wavelength","ethernet","dia","naas_on_demand"], [30,35,20,15] if c.naas_flag else [35,40,25,0])[0]
        scheme = rng.choices(["dual_path","protected","unprotected"], [15,35,50])[0]
        ltype = "wavelength" if stype in ("wavelength","naas_on_demand") else "port"
        prim = rng.choice(lr_pool[ltype])
        prot = pick_protect(ltype, prim) if scheme != "unprotected" else None
        status = rng.choices(["active","pending","disconnected"], [94,4,2])[0]
    svc_rows.append((int(src.service_id), int(cid), spans[prim.span_id].corridor_id, stype, scheme, int(prim.resource_id),
                     int(prot.resource_id) if prot else None, src.monthly_recurring_charge, status, int(src.party_id) if src.party_id is not None else None,
                     f"{CAT}.tmf_service.service", str(src.service_id), "MIXED", RUN_ID))
    lat = None
    try: lat = float(sla.latency_threshold_milliseconds)
    except (TypeError, ValueError): pass
    if lat is None or not (0 < lat <= 1000): lat = round(rng.uniform(5, 60), 1)   # source values are unparsable strings
    p = float(sla.penalty_cap_percentage) if sla.penalty_cap_percentage is not None else None
    credit = round(p, 2) if p is not None and 0 < p <= 100 else round(5 + ((p or 0) % 20), 2)  # source is not on a 0-100 scale
    if credit <= 0: credit = 5.0
    avail = {"dual_path": 99.999, "protected": 99.99, "unprotected": 99.9}[scheme]
    sla_rows.append((int(sla.service_level_agreement_id), int(src.service_id), int(cid), float(lat), avail, float(credit),
                     int(sla.customer_id) if sla.customer_id is not None else None,
                     f"{CAT}.tmf_customer.service_level_agreement", str(sla.service_level_agreement_id), "MIXED", RUN_ID))
svc_schema = T.StructType([T.StructField(n, t) for n, t in [("service_id",T.LongType()),("customer_id",T.LongType()),("corridor_id",T.StringType()),
    ("service_type",T.StringType()),("protection_scheme",T.StringType()),("primary_logical_resource_id",T.LongType()),("protect_logical_resource_id",T.LongType()),
    ("mrr",T.DecimalType(18,2)),("status",T.StringType()),("source_party_id",T.LongType()),("source_table",T.StringType()),("source_id",T.StringType()),
    ("origin",T.StringType()),("simulation_run_id",T.StringType())]])
sla_schema = T.StructType([T.StructField(n, t) for n, t in [("sla_id",T.LongType()),("service_id",T.LongType()),("customer_id",T.LongType()),
    ("latency_target_ms",T.DoubleType()),("availability_target_pct",T.DoubleType()),("credit_pct",T.DoubleType()),("source_customer_id",T.LongType()),
    ("source_table",T.StringType()),("source_id",T.StringType()),("origin",T.StringType()),("simulation_run_id",T.StringType())]])
publish("service", spark.createDataFrame(svc_rows, svc_schema).orderBy("service_id"),
  "Lakelink Fiber customer services (CFS), grain = one row per service_id, 1,500 rows, origin MIXED (EXTEND tmf_service.service, 1,500 distinct active source services). "
  "Injected anomaly SRLG-88: each of the 12 dual_path_flag customers has one active dual_path wavelength service with primary on S-4471 (CHI-DEN) and protect on S-4472 (CHI-DEN-D2); no other service touches S-4471/S-4472.",
  {"service_id":"PK. = tmf_service.service.service_id (distinct sample).","customer_id":"FK -> customer.customer_id (twin assignment; source has no customer_id).",
   "corridor_id":"FK -> corridor.corridor_id of the primary path span.","service_type":"wavelength, ethernet, dia or naas_on_demand.",
   "protection_scheme":"dual_path, protected or unprotected.","primary_logical_resource_id":"FK -> logical_resource.resource_id (working path).",
   "protect_logical_resource_id":"FK -> logical_resource.resource_id (protect path, diverse corridor); null when unprotected, required for dual_path.",
   "mrr":"Monthly recurring revenue, USD DECIMAL(18,2), from source monthly_recurring_charge; > 0.","status":"active, pending or disconnected.",
   "source_party_id":"Lineage only: source tmf_service.service.party_id.", **LINEAGE})
publish("service_level_agreement", spark.createDataFrame(sla_rows, sla_schema).orderBy("sla_id"),
  "Lakelink Fiber service level agreements, grain = one row per service (1:1), 1,500 rows, origin MIXED (EXTEND tmf_customer.service_level_agreement). "
  "1,226 SLAs belong to the twin customer in the source; 274 reuse other source SLA ids (source_customer_id keeps the original). Dual Path SRLG-88 services all have credit_pct > 0.",
  {"sla_id":"PK. = tmf_customer.service_level_agreement.service_level_agreement_id.","service_id":"FK UNIQUE -> service.service_id.",
   "customer_id":"FK -> customer.customer_id; equals service.customer_id.","latency_target_ms":"Latency target in ms; source latency_threshold_milliseconds is unparsable, so generated uniformly in 5-60 (seed 4471).",
   "availability_target_pct":"Availability target percent (0-100 scale): dual_path 99.999, protected 99.99, unprotected 99.9.",
   "credit_pct":"SLA credit percent of MRR on breach (0-100 scale). Source penalty_cap_percentage when within 0-100, otherwise 5 + (value mod 20).",
   "source_customer_id":"Lineage only: customer_id on the source SLA row (differs from customer_id for the 274 reused SLAs).", **LINEAGE})

In [0]:
# Contract s2.5: high_value_flag = top decile (12 of 120) by total service MRR, ties broken by customer_id,
# OR copper_wc_id = 'IA-WC-01' (the 3 Iowa customers). Single in-place UPDATE keeps table/column comments and tags.
spark.sql(f"""
UPDATE {FQ('customer')} c SET high_value_flag = (
  coalesce(c.copper_wc_id, '') = 'IA-WC-01' OR c.customer_id IN (
    SELECT customer_id FROM (
      SELECT cu.customer_id, row_number() OVER (ORDER BY coalesce(sum(s.mrr),0) DESC, cu.customer_id) rk
      FROM {FQ('customer')} cu LEFT JOIN {FQ('service')} s ON s.customer_id = cu.customer_id
      GROUP BY cu.customer_id) WHERE rk <= 12))
""")
display(spark.sql(f"""
SELECT count(*) n, count_if(high_value_flag) hv, count_if(high_value_flag AND copper_wc_id='IA-WC-01') hv_iowa,
       count_if(copper_wc_id='IA-WC-01' AND NOT high_value_flag) iowa_not_hv, count_if(high_value_flag IS NULL) hv_null
FROM {FQ('customer')}"""))

In [0]:
# GENERATE (no span-hourly telemetry in cdm_tmforum). 600 spans x 2,160 hours. Hash-based noise => reproducible regardless of partitioning.
# DEN-SLC (binding, contract s2.8): u(t)=55+30*ln(1+k t)/ln(1+90k), k=0.108783; other corridors linear baseline_util_pct -> end_util_pct
import math
K = 0.108783
def unif(*cols, salt):
    h = F.xxhash64(F.lit(SEED), F.lit(salt), *cols)
    return (F.pmod(h, F.lit(1_000_000_007)).cast("double") + 0.5) / 1_000_000_007.0
def gauss(*cols, salt):  # Box-Muller
    return F.sqrt(-2.0 * F.log(unif(*cols, salt=salt + "a"))) * F.cos(2 * math.pi * unif(*cols, salt=salt + "b"))

spans = (spark.table(FQ("physical_resource")).where("type='span'")
         .select(F.col("resource_id").alias("span_id"), "corridor_id", "length_km")
         .join(spark.table(FQ("corridor")).select("corridor_id", "baseline_util_pct", "end_util_pct"), "corridor_id"))
hours = spark.range(0, 24 * 90).select(F.col("id").alias("h"),
    F.timestamp_seconds(F.unix_timestamp(F.lit("2026-08-18 00:00:00")) + F.col("id") * 3600).alias("ts"))
df = spans.crossJoin(hours)
t = F.col("h") / 24.0
den = F.col("corridor_id") == "DEN-SLC"
trend = F.when(den, 55 + 30 * F.log1p(K * t) / math.log1p(90 * K)) \
         .otherwise(F.col("baseline_util_pct") + (F.col("end_util_pct") - F.col("baseline_util_pct")) * t / 90.0)
span_off = F.when(den, F.lit(0.0)).otherwise((unif("span_id", salt="off") - 0.5) * 4.0)   # +/-2 pts per span
diurnal = 4.0 * F.sin(2 * math.pi * ((F.col("h") % 24) - 8) / 24.0)                      # peak ~14:00 UTC
util = F.least(F.lit(100.0), F.greatest(F.lit(0.0), trend + span_off + diurnal + 3.0 * gauss("span_id", "h", salt="u")))
df = df.withColumn("utilization_pct", F.round(util, 2))
lat = 0.5 + 0.005 * F.col("length_km") + 2.0 * F.greatest(F.lit(0.0), F.col("utilization_pct") - 70) / 30.0 \
      + 0.15 * F.abs(gauss("span_id", "h", salt="l"))
margin = 4.0 + 6.0 * unif("span_id", salt="m") - 0.004 * F.col("h") / 24.0 * 10 / 9 * 0.9 + 0.3 * gauss("span_id", "h", salt="o")
tel = df.select("ts", "span_id", "corridor_id", "utilization_pct",
                F.round(lat, 3).alias("latency_ms"),
                F.round(F.greatest(F.lit(0.5), margin), 2).alias("optical_margin_db"),
                F.lit("SYNTHETIC").alias("origin"), F.lit(RUN_ID).alias("simulation_run_id"))
publish("telemetry_span_hourly", tel,
  "Lakelink Fiber Network Scenario Twin: hourly span telemetry (SID Usage), 600 spans x 24 h x 90 days (2026-08-18 to 2026-11-15 UTC). SYNTHETIC. Injected anomaly: DEN-SLC RapidRoute corridor 7-day mean utilization follows an AI-growth curve 55% -> 85%, crossing 95% around 2027-03-16. Grain: one row per span per hour.",
  {"ts": "Hour start timestamp (UTC), 2026-08-18 00:00 to 2026-11-15 23:00.",
   "span_id": "FK -> physical_resource.resource_id (type span).",
   "corridor_id": "Corridor of the span, denormalized from physical_resource.corridor_id (FK -> corridor.corridor_id).",
   "utilization_pct": "Hourly mean span utilization percent (0-100); trend + diurnal amplitude 4 + Gaussian noise sd 3.",
   "latency_ms": "Hourly mean one-way latency in ms (> 0): propagation from length_km plus a queueing term above 70% utilization.",
   "optical_margin_db": "Optical signal margin in dB (typical 2-12, floor 0.5). Lower is worse.",
   "origin": "Data origin: SYNTHETIC.", "simulation_run_id": "Generator run id (nst-sim-001)."})

# Contract checks (s2.8)
T = FQ("telemetry_span_hourly")
display(spark.sql(f"""
SELECT 'rows' k, count(*) v FROM {T}
UNION ALL SELECT 'dup_span_ts', count(*) - count(DISTINCT span_id, ts) FROM {T}
UNION ALL SELECT 'span_fk_orphan', count(*) FROM {T} t LEFT ANTI JOIN (SELECT resource_id FROM {FQ('physical_resource')} WHERE type='span') p ON t.span_id=p.resource_id
UNION ALL SELECT 'util_out_of_range', count_if(utilization_pct<0 OR utilization_pct>100) FROM {T}
UNION ALL SELECT 'latency_nonpos', count_if(latency_ms<=0) FROM {T}"""))
display(spark.sql(f"""WITH dly AS (SELECT corridor_id, to_date(ts) d, avg(utilization_pct) u FROM {T} GROUP BY 1,2),
r AS (SELECT corridor_id, d, avg(u) OVER (PARTITION BY corridor_id ORDER BY d ROWS 6 PRECEDING) u7 FROM dly)
SELECT corridor_id, round(max(u7),2) max_7d_mean, round(max_by(u7,d),2) last_7d_mean FROM r WHERE d>='2026-08-24' GROUP BY 1 ORDER BY 2 DESC"""))

In [0]:
# Contract s2.12. 52 reused WCs (EXTEND copper_retirement.wire_center_boundary + gold_wire_center_scorecard; source has no IA)
# + 8 SYNTHETIC Iowa WCs. IA-WC-01 is the single row matching acceptance query 3 (hv=3, target 2027-05-01).
import datetime
WCB = f"{CAT}.copper_retirement.wire_center_boundary"
SC  = f"{CAT}.copper_retirement.gold_wire_center_scorecard"
reused = (spark.table(WCB).alias("b")
  .join(spark.table(SC).select("wire_center_id","retirement_readiness_score","retirement_priority_rank").alias("s"),"wire_center_id","left")
  .where("b.state_code <> 'IA'")
  .orderBy(F.col("retirement_priority_rank").asc_nulls_last(), "wire_center_id").limit(52)
  .select(F.col("wire_center_id").alias("wc_id"),
          F.concat(F.lit("Lakelink Fiber "), F.col("wire_center_name"), F.lit(" WC")).alias("wc_name"),
          F.col("state_code").alias("state"), "clli_code",
          F.round(F.greatest(F.col("area_sq_km")*1.8 + F.col("estimated_copper_pairs")*0.05, F.lit(0.5)),1).alias("copper_miles"),
          F.coalesce(F.col("estimated_copper_pairs"), F.lit(0)).cast("int").alias("active_circuits"),
          F.expr("add_months(DATE'2027-07-01', 3*(greatest(coalesce(annual_retire_priority,1),1)-1))").alias("decommission_target"),
          F.col("retirement_readiness_score").cast("double").alias("retirement_readiness_score"),
          F.lit(WCB).alias("source_table"), F.col("wire_center_id").alias("source_id"), F.lit("MIXED").alias("origin")))
rng = random.Random(SEED)
IA = [("IA-WC-01","Ames Central","AMESIAAC","2027-05-01"),("IA-WC-02","Boone North","BOONIABN","2027-09-01"),
      ("IA-WC-03","Marshalltown East","MRTWIAME","2027-10-01"),("IA-WC-04","Nevada Junction","NEVDIANJ","2028-01-01"),
      ("IA-WC-05","Ankeny West","ANKNIAAW","2027-03-01"),("IA-WC-06","Story City","STCYIASC","2028-04-01"),
      ("IA-WC-07","Grinnell South","GRNLIAGS","2027-08-01"),("IA-WC-08","Newton Plains","NWTNIANP","2028-06-01")]
ia_rows = [(w, f"Lakelink Fiber {n} WC", "IA", c, round(rng.uniform(40,220),1), rng.randint(150,1800),
            datetime.date.fromisoformat(d), round(rng.uniform(55,90) if w=="IA-WC-01" else rng.uniform(25,70),1),
            None, None, "SYNTHETIC") for w,n,c,d in IA]
ia = spark.createDataFrame(ia_rows, "wc_id string, wc_name string, state string, clli_code string, copper_miles double, active_circuits int, decommission_target date, retirement_readiness_score double, source_table string, source_id string, origin string")
# high_value_customers reconciles with customer.copper_wc_id AND high_value_flag
hv = (spark.table(FQ("customer")).where("high_value_flag AND copper_wc_id IS NOT NULL")
      .groupBy(F.col("copper_wc_id").alias("wc_id")).agg(F.count("*").cast("int").alias("high_value_customers")))
wc = (reused.unionByName(ia).join(hv, "wc_id", "left")
      .withColumn("high_value_customers", F.coalesce("high_value_customers", F.lit(0)))
      .withColumn("simulation_run_id", F.lit(RUN_ID))
      .select("wc_id","wc_name","state","clli_code","copper_miles","active_circuits","high_value_customers",
              "decommission_target","retirement_readiness_score","source_table","source_id","origin","simulation_run_id"))
publish("copper_wire_center", wc,
  "Lakelink Fiber copper wire centers in scope for decommissioning sequencing (60 rows: 52 reused from copper_retirement.wire_center_boundary + gold_wire_center_scorecard, 8 synthetic Iowa). Injected anomaly: IA-WC-01 is the only Iowa WC with high-value customers (3) and a decommission target before 2027-06-01. Grain: one row per wire center. PK wc_id.",
  {"wc_id":"PK. Reused rows: source wire_center_id. Generated Iowa rows: IA-WC-##.",
   "wc_name":"Lakelink Fiber display name of the wire center.",
   "state":"2-letter US state code. Iowa (IA) rows are all SYNTHETIC.",
   "clli_code":"8-character CLLI code; from source for reused rows, synthetic for IA rows.",
   "copper_miles":"Copper plant route miles served by the wire center (> 0). Reused: area_sq_km*1.8 + estimated_copper_pairs*0.05; IA: generated.",
   "active_circuits":"Active copper circuits (>= 0). Reused: source estimated_copper_pairs; IA: generated.",
   "high_value_customers":"Count of twin customers with copper_wc_id = wc_id AND high_value_flag = TRUE (reconciled at generation).",
   "decommission_target":"Planned copper decommission date. Reused: 2027-07-01 + (annual_retire_priority-1) quarters. IA-WC-01: 2027-05-01.",
   "retirement_readiness_score":"0-100 readiness score. Reused: gold_wire_center_scorecard.retirement_readiness_score; IA: generated.",
   **LINEAGE})
# Contract checks (all 0 except rows=60, acceptance_q3_rows=1)
display(spark.sql(f"""
SELECT 'rows' k, count(*) v FROM {FQ('copper_wire_center')}
UNION ALL SELECT 'acceptance_q3_rows', count(*) FROM {FQ('copper_wire_center')} WHERE state='IA' AND high_value_customers>0 AND decommission_target<'2027-06-01'
UNION ALL SELECT 'hv_mismatch', count(*) FROM {FQ('copper_wire_center')} w LEFT JOIN (SELECT copper_wc_id, count_if(high_value_flag) n FROM {FQ('customer')} GROUP BY 1) h ON w.wc_id=h.copper_wc_id WHERE w.high_value_customers <> coalesce(h.n,0)
UNION ALL SELECT 'customer_wc_fk_orphan', count(*) FROM {FQ('customer')} x LEFT ANTI JOIN {FQ('copper_wire_center')} w ON x.copper_wc_id=w.wc_id WHERE x.copper_wc_id IS NOT NULL
UNION ALL SELECT 'reused_src_orphan', count(*) FROM (SELECT wc_id FROM {FQ('copper_wire_center')} WHERE origin='MIXED') x LEFT ANTI JOIN {WCB} b ON x.wc_id=b.wire_center_id"""))

In [0]:
# Contract s2.9-2.11. alarm + trouble_ticket EXTEND (origin MIXED); maintenance_window GENERATE (origin SYNTHETIC).
import datetime as dt
W0, W1 = "2026-08-18 00:00:00", "2026-11-17 00:00:00"

# ---- alarm: seed-4471 hash sample of source alarms on the 600 twin spans; raised_at linearly rescaled into the window, durations kept
ALARM_SRC = f"{CAT}.provable_resilience_intelligence.resource_alarm"
alarm_df = spark.sql(f"""
WITH tw AS (SELECT resource_id, source_id FROM {FQ('physical_resource')} WHERE type='span'),
src AS (SELECT a.alarm_id, tw.resource_id, a.alarm_type, a.severity, a.raised_at, a.cleared_at
        FROM {ALARM_SRC} a JOIN tw ON a.span_id = tw.source_id
        ORDER BY xxhash64(a.alarm_id, {SEED}), a.alarm_id LIMIT 50000),
b AS (SELECT min(raised_at) s0, max(raised_at) s1 FROM src),
m AS (SELECT src.*, timestamp'{W0}' + make_dt_interval(0,0,0,
        CAST((unix_timestamp(raised_at) - unix_timestamp(b.s0)) * (unix_timestamp(timestamp'{W1}') - unix_timestamp(timestamp'{W0}') - 3600)
             / (unix_timestamp(b.s1) - unix_timestamp(b.s0)) AS DECIMAL(18,0))) AS ts
      FROM src CROSS JOIN b)
SELECT alarm_id, ts,
  CASE WHEN cleared_at IS NULL OR ts + (cleared_at - raised_at) >= timestamp'{W1}' THEN NULL ELSE ts + (cleared_at - raised_at) END AS cleared_ts,
  resource_id, severity, alarm_type,
  '{ALARM_SRC}' AS source_table, alarm_id AS source_id, 'MIXED' AS origin, '{RUN_ID}' AS simulation_run_id
FROM m""")
publish("alarm", alarm_df,
  "Lakelink Fiber Network Scenario Twin: span alarms. Grain: 1 row per alarm (50,000). EXTEND of provable_resilience_intelligence.resource_alarm (deterministic seed-4471 hash sample of alarms on the 600 twin spans); raised_at linearly rescaled into the demo window [2026-08-18, 2026-11-17), durations preserved; origin MIXED; run nst-sim-001. No injected anomaly.",
  {"alarm_id": "PK. Alarm id; equals provable_resilience_intelligence.resource_alarm.alarm_id.",
   "ts": "Alarm raised time (UTC), source raised_at rescaled into the demo window.",
   "cleared_ts": "Alarm cleared time (UTC) = ts + source clear duration; null if uncleared or the clear falls at/after the window end 2026-11-17 (still open at anchor).",
   "resource_id": "FK -> physical_resource.resource_id (span), remapped from the source span_id via physical_resource.source_id.",
   "severity": "Alarm severity: critical, major, minor, warning (from source).",
   "alarm_type": "Alarm type from source: OSNR_DEGRADE, BER_HIGH, LOS, AMP_FAIL, POWER_FAIL, FIBER_CUT.", **LINEAGE})

# ---- trouble_ticket: seed-4471 sample of service_problem (rows whose service_id is an active twin service first; others re-assigned by hash).
# ~35% linked to a critical/major alarm on the service primary span. Source timestamps are incoherent, so times are synthesized in the window.
SP_SRC = f"{CAT}.tmf_service.service_problem"
tt_df = spark.sql(f"""
WITH svc AS (SELECT s.service_id, lr.span_id, row_number() OVER (ORDER BY s.service_id) - 1 AS rn
             FROM {FQ('service')} s JOIN {FQ('logical_resource')} lr ON s.primary_logical_resource_id = lr.resource_id WHERE s.status = 'active'),
n AS (SELECT count(*) AS n FROM svc),
sp AS (SELECT p.service_problem_id, p.service_id AS src_service_id, p.root_cause_code, (p.service_id IN (SELECT service_id FROM svc)) AS svc_match
       FROM {SP_SRC} p ORDER BY svc_match DESC, xxhash64(p.service_problem_id, {SEED}), p.service_problem_id LIMIT 20000),
asg AS (SELECT sp.*, CASE WHEN svc_match THEN src_service_id END AS keep_id,
               pmod(xxhash64(service_problem_id, {SEED}, 'svc'), n.n) AS pick_rn,
               pmod(xxhash64(service_problem_id, {SEED}, 'link'), 1000) / 1000.0 AS u_link,
               pmod(xxhash64(service_problem_id, {SEED}, 'mttr'), 1000) / 1000.0 AS u_mttr,
               pmod(xxhash64(service_problem_id, {SEED}, 'open'), 1000000) / 1000000.0 AS u_open
        FROM sp CROSS JOIN n),
s2 AS (SELECT a.*, coalesce(a.keep_id, svc.service_id) AS service_id FROM asg a JOIN svc ON svc.rn = a.pick_rn),
s3 AS (SELECT s2.*, svc.span_id FROM s2 JOIN svc ON svc.service_id = s2.service_id),
al AS (SELECT resource_id, sort_array(collect_list(named_struct('ts', ts, 'alarm_id', alarm_id, 'alarm_type', alarm_type))) AS arr
       FROM {FQ('alarm')} WHERE severity IN ('critical','major') AND ts < timestamp'{W1}' - INTERVAL 1 DAY GROUP BY resource_id),
l AS (SELECT s3.*, CASE WHEN s3.u_link < 0.35 AND al.arr IS NOT NULL
                        THEN element_at(al.arr, CAST(pmod(xxhash64(s3.service_problem_id, {SEED}, 'pick'), size(al.arr)) + 1 AS INT)) END AS a
      FROM s3 LEFT JOIN al ON al.resource_id = s3.span_id),
rc AS (SELECT l.*, CASE
         WHEN a IS NOT NULL THEN CASE a.alarm_type WHEN 'FIBER_CUT' THEN 'fiber_cut' WHEN 'LOS' THEN 'fiber_cut'
              WHEN 'OSNR_DEGRADE' THEN 'optical_degradation' WHEN 'BER_HIGH' THEN 'optical_degradation'
              WHEN 'AMP_FAIL' THEN 'equipment' WHEN 'POWER_FAIL' THEN 'power' ELSE 'unknown' END
         ELSE CASE root_cause_code WHEN 'external_network' THEN 'fiber_cut' WHEN 'capacity_exhaustion' THEN 'optical_degradation'
              WHEN 'hardware_failure' THEN 'equipment' WHEN 'vendor_issue' THEN 'equipment' WHEN 'power_failure' THEN 'power'
              WHEN 'configuration_error' THEN 'config' WHEN 'software_defect' THEN 'config' WHEN 'human_error' THEN 'config'
              WHEN 'security_attack' THEN 'config' WHEN 'other' THEN 'customer_premises' ELSE 'unknown' END END AS root_cause
       FROM l),
mt AS (SELECT rc.*, CAST(CASE root_cause WHEN 'fiber_cut' THEN 240 + u_mttr*480 WHEN 'optical_degradation' THEN 60 + u_mttr*240
              WHEN 'equipment' THEN 120 + u_mttr*360 WHEN 'power' THEN 60 + u_mttr*180 WHEN 'config' THEN 30 + u_mttr*150
              WHEN 'customer_premises' THEN 60 + u_mttr*300 ELSE 60 + u_mttr*420 END AS INT) AS mttr0 FROM rc),
op AS (SELECT mt.*, CASE WHEN a IS NOT NULL
              THEN a.ts + make_dt_interval(0, 0, CAST(5 + pmod(xxhash64(service_problem_id, {SEED}, 'lag'), 26) AS INT), 0)
              ELSE timestamp'{W0}' + make_dt_interval(0,0,0, CAST(u_open * (unix_timestamp(timestamp'{W1}') - unix_timestamp(timestamp'{W0}') - mttr0*60 - 3600) AS DECIMAL(18,0)))
              END AS opened_ts FROM mt),
fin AS (SELECT op.*, LEAST(mttr0, CAST(floor((unix_timestamp(timestamp'{W1}') - unix_timestamp(opened_ts)) / 60) AS INT) - 1) AS mttr_min FROM op)
SELECT service_problem_id AS ticket_id, service_id, a.alarm_id AS alarm_id, opened_ts,
       opened_ts + make_dt_interval(0, 0, mttr_min, 0) AS resolved_ts, mttr_min, root_cause, src_service_id AS source_service_id,
       '{SP_SRC}' AS source_table, CAST(service_problem_id AS STRING) AS source_id, 'MIXED' AS origin, '{RUN_ID}' AS simulation_run_id
FROM fin""")
publish("trouble_ticket", tt_df,
  "Lakelink Fiber Network Scenario Twin: service trouble tickets. Grain: 1 row per ticket (20,000). EXTEND of tmf_service.service_problem (seed-4471 sample; source problems whose service_id is an active twin service kept first, others re-assigned deterministically to an active twin service). ~35% linked to a critical/major alarm on the service primary span (opened 5-30 min after the alarm). Timestamps synthesized inside the demo window [2026-08-18, 2026-11-17) because source timestamps are not coherent; origin MIXED; run nst-sim-001. No injected anomaly.",
  {"ticket_id": "PK. Ticket id; equals tmf_service.service_problem.service_problem_id.",
   "service_id": "FK -> service.service_id (active twin services only).",
   "alarm_id": "FK -> alarm.alarm_id (nullable). Set when the ticket was triggered by a critical/major alarm on the service primary span.",
   "opened_ts": "Ticket opened time (UTC) within the demo window.",
   "resolved_ts": "Ticket resolved time (UTC) = opened_ts + mttr_min; always < 2026-11-17.",
   "mttr_min": "Mean-time-to-repair for this ticket in minutes = minutes(resolved_ts - opened_ts).",
   "root_cause": "Normalized root cause: fiber_cut, optical_degradation, equipment, power, config, customer_premises, unknown. From the linked alarm_type, else mapped from source root_cause_code.",
   "source_service_id": "Lineage: original tmf_service.service_problem.service_id before re-assignment to a twin service.", **LINEAGE})

# ---- maintenance_window: GENERATE. 199 windows on non-S-4471 spans + the injected S-4471 window (scheduled, 2026-11-20 06:00 UTC).
# Completed starts in [2026-08-18, 2026-11-15]; scheduled starts in [2026-11-21, 2027-02-27] => S-4471 is the next scheduled window after the anchor.
rng = random.Random(SEED + 11)
spans = [r.resource_id for r in spark.table(FQ("physical_resource")).where("type='span' AND resource_id <> 'S-4471'").orderBy("resource_id").collect()]
WT = ["splice", "reroute", "upgrade", "repair"]
def pick_start(d0, d1):
    day = d0 + dt.timedelta(days=rng.randrange((d1 - d0).days + 1))
    return dt.datetime.combine(day, dt.time(rng.choice([0, 1, 2, 3, 4, 5, 6, 22, 23])))
rows = []
for i in range(199):
    sched = i >= 120
    st = pick_start(dt.date(2026, 11, 21), dt.date(2027, 2, 27)) if sched else pick_start(dt.date(2026, 8, 18), dt.date(2026, 11, 15))
    rows.append([rng.choice(spans), st, st + dt.timedelta(hours=rng.randint(2, 8)), "scheduled" if sched else "completed", rng.choice(WT)])
s0 = dt.datetime(2026, 11, 20, 6, 0)
rows.append(["S-4471", s0, s0 + dt.timedelta(hours=4), "scheduled", "splice"])
rows.sort(key=lambda r: (r[1], r[0]))
mw_df = spark.createDataFrame([(f"MW-{i+1:04d}", *r, None, None, "SYNTHETIC", RUN_ID) for i, r in enumerate(rows)],
    "window_id STRING, span_id STRING, start_ts TIMESTAMP, end_ts TIMESTAMP, status STRING, work_type STRING, source_table STRING, source_id STRING, origin STRING, simulation_run_id STRING")
publish("maintenance_window", mw_df,
  "Lakelink Fiber Network Scenario Twin: planned maintenance windows on fiber spans. Grain: 1 row per window (200). GENERATE (no source in cdm_tmforum); origin SYNTHETIC; run nst-sim-001. INJECTED ANOMALY: exactly one window on S-4471 (SRLG-88), status scheduled, start 2026-11-20 06:00 UTC - the next scheduled window after the 2026-11-16 anchor.",
  {"window_id": "PK. Maintenance window id MW-0001..MW-0200, numbered by start_ts.",
   "span_id": "FK -> physical_resource.resource_id (type span).",
   "start_ts": "Window start (UTC). Range 2026-08-18 to 2027-02-28.",
   "end_ts": "Window end (UTC), exclusive; end_ts > start_ts, duration 2-8 hours.",
   "status": "completed (started before the 2026-11-16 anchor) or scheduled (future).",
   "work_type": "Planned work type: splice, reroute, upgrade, repair.", **LINEAGE})
print("Published without telco_project tag:", UNTAGGED)

In [0]:
# Expected: alarm_rows=50000, tt_rows=20000, mw_rows=200, s4471_windows=1, next_sched_is_s4471=1; every other value 0
display(spark.sql(f"""
SELECT 'alarm_rows' k, count(*) v FROM {FQ('alarm')}
UNION ALL SELECT 'alarm_pk_dup', count(*) - count(DISTINCT alarm_id) FROM {FQ('alarm')}
UNION ALL SELECT 'alarm_span_orphan', count(*) FROM {FQ('alarm')} a LEFT ANTI JOIN {FQ('physical_resource')} p ON a.resource_id=p.resource_id AND p.type='span'
UNION ALL SELECT 'alarm_src_orphan', count(*) FROM {FQ('alarm')} a LEFT ANTI JOIN {CAT}.provable_resilience_intelligence.resource_alarm r ON a.alarm_id=r.alarm_id
UNION ALL SELECT 'alarm_out_of_window', count_if(ts < timestamp'2026-08-18' OR ts >= timestamp'2026-11-17' OR cleared_ts < ts) FROM {FQ('alarm')}
UNION ALL SELECT 'tt_rows', count(*) FROM {FQ('trouble_ticket')}
UNION ALL SELECT 'tt_pk_dup', count(*) - count(DISTINCT ticket_id) FROM {FQ('trouble_ticket')}
UNION ALL SELECT 'tt_service_orphan', count(*) FROM {FQ('trouble_ticket')} t LEFT ANTI JOIN {FQ('service')} s ON t.service_id=s.service_id
UNION ALL SELECT 'tt_alarm_orphan', count(*) FROM {FQ('trouble_ticket')} t LEFT ANTI JOIN {FQ('alarm')} a ON t.alarm_id=a.alarm_id WHERE t.alarm_id IS NOT NULL
UNION ALL SELECT 'tt_src_orphan', count(*) FROM {FQ('trouble_ticket')} t LEFT ANTI JOIN {CAT}.tmf_service.service_problem p ON t.ticket_id=p.service_problem_id
UNION ALL SELECT 'tt_bad_interval', count_if(resolved_ts < opened_ts OR opened_ts < timestamp'2026-08-18' OR resolved_ts >= timestamp'2026-11-17' OR mttr_min <> CAST((unix_timestamp(resolved_ts)-unix_timestamp(opened_ts))/60 AS INT)) FROM {FQ('trouble_ticket')}
UNION ALL SELECT 'mw_rows', count(*) FROM {FQ('maintenance_window')}
UNION ALL SELECT 'mw_span_orphan', count(*) FROM {FQ('maintenance_window')} m LEFT ANTI JOIN {FQ('physical_resource')} p ON m.span_id=p.resource_id AND p.type='span'
UNION ALL SELECT 'mw_bad_duration', count_if((unix_timestamp(end_ts)-unix_timestamp(start_ts))/3600 NOT BETWEEN 2 AND 8) FROM {FQ('maintenance_window')}
UNION ALL SELECT 's4471_windows', count_if(span_id='S-4471') FROM {FQ('maintenance_window')}
UNION ALL SELECT 'next_sched_is_s4471', CAST(min_by(span_id, start_ts) = 'S-4471' AS INT) FROM {FQ('maintenance_window')} WHERE status='scheduled' AND start_ts >= timestamp'2026-11-16'"""))

## Next sections
* **P1-GEN-TOPOLOGY**: physical_resource (800; S-4471 on CHI-DEN, S-4472 on CHI-DEN-D2, both SRLG-88), logical_resource (2,500)
* **P1-GEN-SERVICE**: service + service_level_agreement (1,500 each); route each dual_path_flag customer primary over S-4471 and protect over S-4472; then set high_value_flag for top-decile MRR customers
* **P1-GEN-TELEMETRY**: telemetry_span_hourly (1,296,000; DEN-SLC curve)
* **P1-GEN-OPS**: alarm, trouble_ticket, maintenance_window (S-4471 window 2026-11-20 06:00 UTC)
* **P1-GEN-COPPER**: copper_wire_center (60; IA-WC-01 = the flagged Iowa WC, high_value_customers=3, target 2027-05-01)